In [ ]:
# ============================================================
# STEP 3: IN-SAMPLE AR(1)-GARCH-TYPE MODEL ESTIMATION
# ============================================================

from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path("data/processed")
OUTPUT_DIR = Path("outputs")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_SUMMARY_DIR = OUTPUT_DIR / "model_summaries"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

RETURNS_PATH = PROCESSED_DIR / "vietnam_size_indices_log_returns_common_20141121_20251231.csv"

# Fallback nếu chạy trong ChatGPT sandbox
if not RETURNS_PATH.exists():
    RETURNS_PATH = Path("/mnt/data/vietnam_size_indices_log_returns_common_20141121_20251231.csv")

# ------------------------------------------------------------
# Load return data
# ------------------------------------------------------------

returns = pd.read_csv(RETURNS_PATH, parse_dates=["date"])
returns = returns.set_index("date").sort_index()

INDEX_COLUMNS = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]
returns = returns[INDEX_COLUMNS]

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert returns.index.is_monotonic_increasing, "Date index is not sorted."
assert returns.index.duplicated().sum() == 0, "Duplicate dates exist."
assert returns.isna().sum().sum() == 0, "Missing values exist."
assert np.isfinite(returns.to_numpy()).all(), "Infinite values exist."

print("Step 3 input data loaded.")
print("Sample:", returns.index.min().date(), "to", returns.index.max().date())
print("Shape:", returns.shape)
print("Return scale check:")
display(returns.describe().T)

In [ ]:
# ============================================================
# 2. MODEL SPECIFICATIONS
# ============================================================

MODEL_SPECS = {
    "AR1_GARCH11_Normal": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "normal"
    },

    "AR1_GARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "t"
    },

    "AR1_GARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "skewt"
    },

    "AR1_GJR_GARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_GJR_GARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    },

    "AR1_EGARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "EGARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_EGARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "EGARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    },

    "AR1_APARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "APARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_APARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "APARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    }
}

print("Model specifications:")
for name, spec in MODEL_SPECS.items():
    print(name, spec)

In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def safe_get_param(params, names):
    """
    Safely extract a parameter from arch result params.
    """
    for name in names:
        if name in params.index:
            return params[name]
    return np.nan


def get_ar1_param(params):
    """
    Extract AR(1) coefficient from arch params.
    Parameter name is usually like 'VN30[1]' or 'None[1]'.
    """
    candidates = []

    for name in params.index:
        if name.endswith("[1]"):
            if not any(x in name.lower() for x in ["alpha", "beta", "gamma"]):
                candidates.append(name)

    if len(candidates) > 0:
        return params[candidates[0]]

    return np.nan


def compute_garch_persistence(params, model_name):
    alpha = safe_get_param(params, ["alpha[1]"])
    beta = safe_get_param(params, ["beta[1]"])
    gamma = safe_get_param(params, ["gamma[1]"])

    if "EGARCH" in model_name or "APARCH" in model_name:
        return np.nan

    if "GJR" in model_name:
        if pd.notna(alpha) and pd.notna(beta) and pd.notna(gamma):
            return alpha + 0.5 * gamma + beta

    if "GARCH11" in model_name and "GJR" not in model_name:
        if pd.notna(alpha) and pd.notna(beta):
            return alpha + beta

    return np.nan


def fit_arch_model(y, index_name, model_name, spec):
    """
    Fit one AR(1)-GARCH-type model.
    """
    am = arch_model(
        y,
        mean=spec["mean"],
        lags=spec["lags"],
        vol=spec["vol"],
        p=spec["p"],
        o=spec["o"],
        q=spec["q"],
        dist=spec["dist"],
        rescale=False
    )

    res = am.fit(
        update_freq=0,
        disp="off",
        show_warning=False,
        options={"maxiter": 2000}
    )

    return res


def residual_diagnostics(std_resid, lags=[10, 20], arch_lags=10):
    """
    Ljung-Box on standardized residuals,
    Ljung-Box on squared standardized residuals,
    ARCH-LM on standardized residuals.
    """
    std_resid = pd.Series(std_resid).replace([np.inf, -np.inf], np.nan).dropna()
    std_resid_sq = std_resid ** 2

    rows = []

    lb_resid = acorr_ljungbox(std_resid, lags=lags, return_df=True)
    lb_sq = acorr_ljungbox(std_resid_sq, lags=lags, return_df=True)

    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(std_resid, nlags=arch_lags)

    for lag in lags:
        rows.append({
            "lag": lag,
            "lb_stat_std_resid": lb_resid.loc[lag, "lb_stat"],
            "lb_pvalue_std_resid": lb_resid.loc[lag, "lb_pvalue"],
            "lb_stat_squared_std_resid": lb_sq.loc[lag, "lb_stat"],
            "lb_pvalue_squared_std_resid": lb_sq.loc[lag, "lb_pvalue"],
            "arch_lm_lags": arch_lags,
            "arch_lm_stat": lm_stat,
            "arch_lm_pvalue": lm_pvalue,
            "arch_f_stat": f_stat,
            "arch_f_pvalue": f_pvalue,
        })

    return rows

In [ ]:
# ============================================================
# 4. FIT ALL AR(1)-GARCH-TYPE MODELS
# ============================================================

fit_results = {}
summary_rows = []
param_rows = []
diagnostic_rows = []

conditional_volatility_dict = {}
std_residuals_dict = {}
raw_residuals_dict = {}

for index_name in INDEX_COLUMNS:
    y = returns[index_name].dropna()

    fit_results[index_name] = {}

    print(f"\n==============================")
    print(f"Fitting models for {index_name}")
    print(f"==============================")

    for model_name, spec in MODEL_SPECS.items():
        print(f"Running {model_name}...")

        result_key = f"{index_name}__{model_name}"

        try:
            res = fit_arch_model(y, index_name, model_name, spec)

            fit_results[index_name][model_name] = res

            params = res.params
            pvalues = res.pvalues
            tvalues = res.tvalues
            std_err = res.std_err

            converged = (res.convergence_flag == 0)

            # Save text summary
            summary_path = MODEL_SUMMARY_DIR / f"step3_{index_name}_{model_name}.txt"
            with open(summary_path, "w", encoding="utf-8") as f:
                f.write(str(res.summary()))

            # Save fitted series
            conditional_volatility_dict[result_key] = res.conditional_volatility
            std_residuals_dict[result_key] = res.std_resid
            raw_residuals_dict[result_key] = res.resid

            # Model-level summary
            summary_rows.append({
                "index": index_name,
                "model": model_name,
                "mean": spec["mean"],
                "lags": spec["lags"],
                "vol": spec["vol"],
                "p": spec["p"],
                "o": spec["o"],
                "q": spec["q"],
                "distribution": spec["dist"],
                "nobs": res.nobs,
                "num_params": len(params),
                "loglikelihood": res.loglikelihood,
                "aic": res.aic,
                "bic": res.bic,
                "convergence_flag": res.convergence_flag,
                "converged": converged,
                "optimization_message": str(res.optimization_result.message),
                "mu_or_const": safe_get_param(params, ["Const", "mu"]),
                "ar1": get_ar1_param(params),
                "omega": safe_get_param(params, ["omega"]),
                "alpha1": safe_get_param(params, ["alpha[1]"]),
                "gamma1": safe_get_param(params, ["gamma[1]"]),
                "beta1": safe_get_param(params, ["beta[1]"]),
                "delta": safe_get_param(params, ["delta"]),
                "nu": safe_get_param(params, ["nu"]),
                "eta": safe_get_param(params, ["eta"]),
                "lambda": safe_get_param(params, ["lambda"]),
                "approx_persistence": compute_garch_persistence(params, model_name),
            })

            # Parameter-level table
            for param_name in params.index:
                param_rows.append({
                    "index": index_name,
                    "model": model_name,
                    "param": param_name,
                    "estimate": params[param_name],
                    "std_error": std_err[param_name],
                    "t_value": tvalues[param_name],
                    "p_value": pvalues[param_name],
                    "significant_5pct": pvalues[param_name] < 0.05
                })

            # Diagnostics
            diagnostics = residual_diagnostics(res.std_resid, lags=[10, 20], arch_lags=10)

            for d in diagnostics:
                d.update({
                    "index": index_name,
                    "model": model_name,
                    "converged": converged
                })
                diagnostic_rows.append(d)

            print(f"Done | converged={converged} | AIC={res.aic:.4f} | BIC={res.bic:.4f}")

        except Exception as e:
            print(f"FAILED | {index_name} | {model_name} | {e}")

            summary_rows.append({
                "index": index_name,
                "model": model_name,
                "mean": spec["mean"],
                "lags": spec["lags"],
                "vol": spec["vol"],
                "p": spec["p"],
                "o": spec["o"],
                "q": spec["q"],
                "distribution": spec["dist"],
                "nobs": len(y),
                "num_params": np.nan,
                "loglikelihood": np.nan,
                "aic": np.nan,
                "bic": np.nan,
                "convergence_flag": np.nan,
                "converged": False,
                "optimization_message": str(e),
                "mu_or_const": np.nan,
                "ar1": np.nan,
                "omega": np.nan,
                "alpha1": np.nan,
                "gamma1": np.nan,
                "beta1": np.nan,
                "delta": np.nan,
                "nu": np.nan,
                "eta": np.nan,
                "lambda": np.nan,
                "approx_persistence": np.nan,
            })

# Convert to dataframes
model_summary = pd.DataFrame(summary_rows)
parameter_table = pd.DataFrame(param_rows)
diagnostics_table = pd.DataFrame(diagnostic_rows)

display(model_summary)
display(parameter_table.head())
display(diagnostics_table.head())

In [ ]:
# ============================================================
# 5. SAVE STEP 3 OUTPUTS
# ============================================================

model_summary_path = TABLE_DIR / "step3_model_summary_ar1_garch_types.csv"
parameter_table_path = TABLE_DIR / "step3_parameter_estimates_ar1_garch_types.csv"
diagnostics_table_path = TABLE_DIR / "step3_residual_diagnostics_ar1_garch_types.csv"

model_summary.to_csv(model_summary_path, index=False)
parameter_table.to_csv(parameter_table_path, index=False)
diagnostics_table.to_csv(diagnostics_table_path, index=False)

# Save fitted conditional volatility, residuals
conditional_volatility = pd.DataFrame(conditional_volatility_dict)
std_residuals = pd.DataFrame(std_residuals_dict)
raw_residuals = pd.DataFrame(raw_residuals_dict)

conditional_volatility.to_csv(TABLE_DIR / "step3_conditional_volatility.csv")
std_residuals.to_csv(TABLE_DIR / "step3_standardized_residuals.csv")
raw_residuals.to_csv(TABLE_DIR / "step3_raw_residuals.csv")

# Save all tables to one Excel workbook
excel_path = TABLE_DIR / "step3_ar1_garch_type_estimation_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    model_summary.to_excel(writer, sheet_name="Model_Summary", index=False)
    parameter_table.to_excel(writer, sheet_name="Parameters", index=False)
    diagnostics_table.to_excel(writer, sheet_name="Residual_Diagnostics", index=False)
    conditional_volatility.to_excel(writer, sheet_name="Conditional_Volatility")
    std_residuals.to_excel(writer, sheet_name="Std_Residuals")

print("Saved Step 3 outputs:")
print(model_summary_path)
print(parameter_table_path)
print(diagnostics_table_path)
print(excel_path)

In [ ]:
# ============================================================
# 6. MODEL RANKING BY AIC AND BIC
# ============================================================

valid_models = model_summary[
    model_summary["converged"] == True
].copy()

aic_ranking = (
    valid_models
    .sort_values(["index", "aic"])
    .groupby("index")
    .head(len(MODEL_SPECS))
    .reset_index(drop=True)
)

bic_ranking = (
    valid_models
    .sort_values(["index", "bic"])
    .groupby("index")
    .head(len(MODEL_SPECS))
    .reset_index(drop=True)
)

best_by_aic = (
    valid_models
    .sort_values(["index", "aic"])
    .groupby("index")
    .head(1)
    .reset_index(drop=True)
)

best_by_bic = (
    valid_models
    .sort_values(["index", "bic"])
    .groupby("index")
    .head(1)
    .reset_index(drop=True)
)

aic_ranking.to_csv(TABLE_DIR / "step3_model_ranking_by_aic.csv", index=False)
bic_ranking.to_csv(TABLE_DIR / "step3_model_ranking_by_bic.csv", index=False)
best_by_aic.to_csv(TABLE_DIR / "step3_best_model_by_aic.csv", index=False)
best_by_bic.to_csv(TABLE_DIR / "step3_best_model_by_bic.csv", index=False)

print("Best models by AIC:")
display(best_by_aic[["index", "model", "distribution", "loglikelihood", "aic", "bic", "converged"]])

print("Best models by BIC:")
display(best_by_bic[["index", "model", "distribution", "loglikelihood", "aic", "bic", "converged"]])

In [ ]:
# ============================================================
# 7. RESIDUAL DIAGNOSTIC PASS / FAIL
# ============================================================

diagnostics_main = diagnostics_table[
    diagnostics_table["lag"] == 10
].copy()

diagnostics_main["std_resid_no_autocorr_5pct"] = diagnostics_main["lb_pvalue_std_resid"] > 0.05
diagnostics_main["squared_std_resid_no_autocorr_5pct"] = diagnostics_main["lb_pvalue_squared_std_resid"] > 0.05
diagnostics_main["no_remaining_arch_5pct"] = diagnostics_main["arch_lm_pvalue"] > 0.05

diagnostics_main["diagnostic_pass"] = (
    diagnostics_main["converged"]
    & diagnostics_main["std_resid_no_autocorr_5pct"]
    & diagnostics_main["squared_std_resid_no_autocorr_5pct"]
    & diagnostics_main["no_remaining_arch_5pct"]
)

diagnostics_main = diagnostics_main.merge(
    model_summary[["index", "model", "aic", "bic", "loglikelihood", "distribution"]],
    on=["index", "model"],
    how="left"
)

diagnostics_main.to_csv(TABLE_DIR / "step3_diagnostic_pass_fail_lag10.csv", index=False)

display(
    diagnostics_main[
        [
            "index",
            "model",
            "distribution",
            "converged",
            "lb_pvalue_std_resid",
            "lb_pvalue_squared_std_resid",
            "arch_lm_pvalue",
            "std_resid_no_autocorr_5pct",
            "squared_std_resid_no_autocorr_5pct",
            "no_remaining_arch_5pct",
            "diagnostic_pass",
            "aic",
            "bic"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# 8. BALANCED CANDIDATE MODEL SET FOR STEP 4 VaR
# ============================================================

# Main principle:
# For cross-index comparison, use the same model set across all indices.
# Do not select only one best model per index before VaR backtesting.

valid_models = model_summary[
    model_summary["converged"] == True
].copy()

# Count how many indices each model converged on
model_convergence_count = (
    valid_models
    .groupby("model")["index"]
    .nunique()
    .reset_index(name="n_indices_converged")
)

model_convergence_count["available_for_all_indices"] = (
    model_convergence_count["n_indices_converged"] == len(INDEX_COLUMNS)
)

display(model_convergence_count)

# Common model set: models that converged for all 4 indices
common_models_for_step4 = model_convergence_count.loc[
    model_convergence_count["available_for_all_indices"],
    "model"
].tolist()

print("Common models available for all indices:")
print(common_models_for_step4)

# Balanced candidate table for Step 4
balanced_candidates_for_step4 = valid_models[
    valid_models["model"].isin(common_models_for_step4)
].copy()

balanced_candidates_for_step4 = balanced_candidates_for_step4.sort_values(
    ["model", "index"]
)

balanced_candidates_for_step4.to_csv(
    TABLE_DIR / "step3_balanced_candidate_models_for_step4_var.csv",
    index=False
)

display(
    balanced_candidates_for_step4[
        [
            "index",
            "model",
            "distribution",
            "loglikelihood",
            "aic",
            "bic",
            "converged",
            "approx_persistence"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# 9. INDEX-SPECIFIC BEST MODEL — SUPPLEMENTARY ONLY
# ============================================================

diagnostics_main = diagnostics_table[
    diagnostics_table["lag"] == 10
].copy()

diagnostics_main["std_resid_no_autocorr_5pct"] = diagnostics_main["lb_pvalue_std_resid"] > 0.05
diagnostics_main["squared_std_resid_no_autocorr_5pct"] = diagnostics_main["lb_pvalue_squared_std_resid"] > 0.05
diagnostics_main["no_remaining_arch_5pct"] = diagnostics_main["arch_lm_pvalue"] > 0.05

diagnostics_main["diagnostic_pass"] = (
    diagnostics_main["converged"]
    & diagnostics_main["std_resid_no_autocorr_5pct"]
    & diagnostics_main["squared_std_resid_no_autocorr_5pct"]
    & diagnostics_main["no_remaining_arch_5pct"]
)

diagnostics_main = diagnostics_main.merge(
    model_summary[
        [
            "index",
            "model",
            "distribution",
            "aic",
            "bic",
            "loglikelihood",
            "converged"
        ]
    ],
    on=["index", "model"],
    how="left",
    suffixes=("", "_summary")
)

diagnostics_main["tail_distribution"] = diagnostics_main["distribution"].isin(["t", "skewt"])

index_specific_ranking = diagnostics_main[
    diagnostics_main["converged"] == True
].copy()

index_specific_ranking = index_specific_ranking.sort_values(
    by=[
        "index",
        "diagnostic_pass",
        "tail_distribution",
        "bic"
    ],
    ascending=[
        True,
        False,
        False,
        True
    ]
)

index_specific_ranking["rank_within_index"] = (
    index_specific_ranking
    .groupby("index")
    .cumcount() + 1
)

index_specific_best = index_specific_ranking[
    index_specific_ranking["rank_within_index"] == 1
].copy()

index_specific_ranking.to_csv(
    TABLE_DIR / "step3_index_specific_model_ranking.csv",
    index=False
)

index_specific_best.to_csv(
    TABLE_DIR / "step3_index_specific_best_model_supplementary.csv",
    index=False
)

display(
    index_specific_best[
        [
            "index",
            "model",
            "distribution",
            "diagnostic_pass",
            "aic",
            "bic",
            "lb_pvalue_std_resid",
            "lb_pvalue_squared_std_resid",
            "arch_lm_pvalue"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# 11. AUTO INTERPRETATION FOR STEP 3
# ============================================================

interpretation_rows = []

for _, row in index_specific_best.iterrows():
    index_name = row["index"]
    model_name = row["model"]

    diag_pass = row["diagnostic_pass"]

    if diag_pass:
        diagnostic_text = (
            "The model passes the main residual diagnostics at lag 10, "
            "suggesting that serial correlation and remaining ARCH effects are adequately controlled."
        )
    else:
        diagnostic_text = (
            "The model does not fully pass the main residual diagnostics at lag 10. "
            "It can still be kept for VaR backtesting, but its adequacy must be judged carefully in Step 4."
        )

    interpretation_rows.append({
        "index": index_name,
        "selected_candidate": model_name,
        "distribution": row["distribution"],
        "aic": row["aic"],
        "bic": row["bic"],
        "diagnostic_pass": diag_pass,
        "interpretation": (
            f"For {index_name}, the selected Step 4 candidate is {model_name}. "
            f"The model uses {row['distribution']} innovations. "
            f"{diagnostic_text} "
            f"Final model choice should be based on out-of-sample VaR backtesting, not only AIC/BIC."
        )
    })

interpretation_table = pd.DataFrame(interpretation_rows)
interpretation_table.to_csv(TABLE_DIR / "step3_interpretation_text.csv", index=False)

for text in interpretation_table["interpretation"]:
    print(text)
    print()

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Optional: make plots publication-friendly
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


INDEX_ORDER = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]


def find_date_column(df):
    candidates = ["Date", "date", "trading_date", "TradingDate", "time", "Time"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No date column found. Available columns: {df.columns.tolist()}")


def standardize_date_index(df):
    df = df.copy()
    date_col = find_date_column(df)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)
    return df


def pick_index_columns(df):
    cols = []
    for idx in INDEX_ORDER:
        exact = [c for c in df.columns if c == idx]
        contains = [c for c in df.columns if idx.lower() in c.lower()]
        if exact:
            cols.append(exact[0])
        elif contains:
            cols.append(contains[0])
        else:
            raise ValueError(f"Cannot find column for {idx}. Available columns: {df.columns.tolist()}")
    return cols


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

In [ ]:
# ============================================================
# Figure 3: Conditional volatility from EGARCH-SkewT
# Input: vietnam_size_indices_log_returns_common_20141121_20251231.csv
# Output:
#   step3_egarch_skewt_conditional_volatility.csv
#   figures/fig_03_conditional_volatility_egarch_skewt.pdf
# ============================================================

from arch import arch_model

ret_path = Path("data/processed/vietnam_size_indices_log_returns_common_20141121_20251231.csv")
returns_df = pd.read_csv(ret_path)
returns_df = standardize_date_index(returns_df)

ret_cols = pick_index_columns(returns_df)
returns_plot = returns_df[ret_cols].copy()
returns_plot.columns = INDEX_ORDER

cond_vol = pd.DataFrame(index=returns_plot.index)

fit_summaries = []

for idx in INDEX_ORDER:
    y = returns_plot[idx].dropna()

    # AR(1)-EGARCH(1,1)-SkewT
    # arch uses p for symmetric term, o for asymmetric term, q for GARCH term.
    model = arch_model(
        y,
        mean="AR",
        lags=1,
        vol="EGARCH",
        p=1,
        o=1,
        q=1,
        dist="skewt",
        rescale=False
    )

    res = model.fit(disp="off", show_warning=False)

    cond_vol.loc[y.index, idx] = res.conditional_volatility

    fit_summaries.append({
        "index": idx,
        "loglik": res.loglikelihood,
        "aic": res.aic,
        "bic": res.bic,
        "converged": res.convergence_flag == 0
    })

cond_vol.to_csv("step3_egarch_skewt_conditional_volatility.csv", index_label="Date")
pd.DataFrame(fit_summaries).to_csv("step3_egarch_skewt_refit_summary_for_figures.csv", index=False)

fig, axes = plt.subplots(4, 1, figsize=(10, 7.2), sharex=True)

for ax, col in zip(axes, INDEX_ORDER):
    ax.plot(
        cond_vol.index,
        cond_vol[col],
        linewidth=0.9,
        zorder=2
    )

    # Keep individual subplot titles
    ax.set_title(col, loc="left", fontsize=10, fontweight="normal")

    ax.set_ylabel("Volatility")

    # Remove grid for report-style figure
    ax.grid(False)

axes[-1].set_xlabel("Date")

# No overall title; use caption in LaTeX instead
fig.tight_layout()

savefig(FIG_DIR / "fig_03_conditional_volatility_egarch_skewt.pdf")